### Weather Cleaning and Organization Notebook
This notebook will focus on converting the raw METAR and SPECI weather data into one usable data frame and exporting it to a CSV file that will be used to perform predictive analysis for on time departures out of San Diego International Airport. This finalized notebook will be used to perform a merge with the on time flight dataset resulting in weather data points for each of the flight information that was obtained.

In [1]:
# Import libraries
import pandas as pd
import re

from datetime import datetime
from metar import Metar

In [ ]:
# Import the data files
df1 = pd.read_csv('raw_weather/KSAN_2023_wx.csv')
df2 = pd.read_csv('raw_weather/KSAN_2024_wx.csv')
df3 = pd.read_csv('raw_weather/KSAN_2025_wx.csv')

# create list of dfs
dfs = [df1, df2, df3]

In [ ]:
# Removing unnecessary rows besides REM which houses the raw METAR data and concatenating the dataframes
metar_dfs = []

for i, df in enumerate(dfs, start=1):
    if 'REM' in df.columns:
        metar_dfs.append(df[['REM']])
    else:
        print('REM column not found in df{i}')

# concatenating the dfs
comb_metar_df = pd.concat(metar_dfs, ignore_index=True)

In [ ]:
comb_metar_df.head()

In [ ]:
comb_metar_df.tail()

In [ ]:
# Filter so that the data only contains METAR data and not extra synthetic information that does not conform to the standard METAR format we will be extracting from
comb_metar_df = comb_metar_df[comb_metar_df['REM'].str.startswith('MET', na=False)]

In [ ]:
# reset the index of the dataframe
comb_metar_df.reset_index(drop=True, inplace=True)

In [ ]:
print(comb_metar_df.head())
print(comb_metar_df.tail())

In [ ]:
# establish a prefix pattern to extract the information before the actual beginning of the METAR data

prefix_pattern = re.compile(
    # the MET pattern and 3 digit code that we do not need
    r'^MET\d{3}'
    # local month
    r'(?P<month>\d{2})/'
    # local day
    r'(?P<day>\d{2})/'
    #local year
    r'(?P<year>\d{2})\s+'
    # local hour
    r'(?P<hour>\d{2}):'
    # local minute
    r'(?P<minute>\d{2}):'
    # local second
    r'(?P<second>\d{2})\s+'
    # rest of the actual metar
    r'(?P<metar>.*)$'
)

In [ ]:
# create function that will parse the prefix data, dropping the MET and the three digit code but making a date and time column 
def parse_prefix(line):
    match = prefix_pattern.match(line)
    if not match:
        return None
    
    local_date = f"{match.group('month')}/{match.group('day')}/{match.group('year')}"
    local_time = f"{match.group('hour')}:{match.group('minute')}:{match.group('second')}"
    metar = match.group('metar')

    return {
        'local_date': local_date,
        'local_time': local_time,
        'metar': metar
    }

In [ ]:
# create function that will parse the raw METAR data.
def parse_metar(metar):
    try:
        parsed = Metar.Metar(metar)

        return {
            'station_id': parsed.station_id,
            'wind_dir_degrees': parsed.wind_dir.value() if parsed.wind_dir else None,
            'wind_speed_kt': parsed.wind_speed.value() if parsed.wind_speed else None,
            'wind_gust_kt': parsed.wind_gust.value() if parsed.wind_gust else None,
            'visibility_statute_mi': parsed.vis.value() if parsed.vis else None,
            'temperature_c': parsed.temp.value() if parsed.temp else None,
            'dewpoint_c': parsed.dewpt.value() if parsed.dewpt else None,
            'altimeter_hpa': parsed.press.value() if parsed.press else None,
        }
    except Exception as e:
        return None

In [ ]:
# tie all the functions together in order to break up each field of every METAR observation into their own columns

parsed_metar_data = []

# process each line of the comb_metar_df
for line in comb_metar_df['REM']:
    prefix_data = parse_prefix(line)
    if prefix_data is None:
        continue
    
    metar_data = parse_metar(prefix_data['metar'])
    if metar_data is None:
        continue

    combined_record = {**prefix_data, **metar_data}
    parsed_metar_data.append(combined_record)

# create the new df from parsed records
organized_metar_df = pd.DataFrame(parsed_metar_data)
organized_metar_df.reset_index(drop=True, inplace=True)

In [ ]:
print(organized_metar_df.head())

In [ ]:
# drop the metar column
organized_metar_df.drop(columns=['metar'], inplace=True)
print(organized_metar_df.head())

In [ ]:
# replace all NaN values in the wind_gust_kt column with 0
organized_metar_df.fillna({'wind_gust_kt':0}, inplace=True)
print(organized_metar_df.head())

In [ ]:
# output the data frame to a csv
organized_metar_df.to_csv('raw_weather/organized_metar_data.csv', index=False)

In [ ]:
%%html
<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>